# Week 22, Day 6: Forge: Week 22 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w022/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("1.", np.linalg.eigvals(np.array([[0, 1], [-4, 0]])), "-> a centre")
print("2.", round(20 / (1 - 0.95), 1), "ms")
print("3. f'(x) = 1 - 2x: f'(0) =", 1 - 2 * 0, "(unstable), f'(1) =", 1 - 2 * 1, "(stable)")

In [ ]:
tau = 10.0
F = lambda u: 1 / (1 + np.exp(-(u - 0.5) / 0.1))
def dec(r, beta, I1=1.0, I2=1.0):
    return np.array([-r[0] + F(I1 - beta * r[1]), -r[1] + F(I2 - beta * r[0])]) / tau
def jac2(r, beta, h=1e-7):
    J = np.zeros((2, 2))
    for j in range(2):
        e = np.zeros(2); e[j] = h; J[:, j] = (dec(r + e, beta) - dec(r - e, beta)) / (2 * h)
    return J
def rk4(f, x, dt):
    k1 = f(x); k2 = f(x + dt / 2 * k1); k3 = f(x + dt / 2 * k2); k4 = f(x + dt * k3)
    return x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
print("flow at (0.5, 0.5) with beta = 1:", dec(np.array([0.5, 0.5]), 1.0))

### Exercise: Part A: a saddle between two winners

Write `sym_fp(beta)`, which finds s solving s = F(1 − beta s) by bisection on [0, 1]. Then refine the guess (0.95, 0.05) with Newton's method (50 steps, using jac2) to get the winner state `win` for beta = 1.

In [ ]:
def sym_fp(beta):
    g = lambda s: -s + F(1 - beta * s)
    lo, hi = 0.0, 1.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if ...: hi = mid
        else: lo = mid
    return (lo + hi) / 2

win = np.array([0.95, 0.05])
for _ in range(50):
    win = ...
for beta in (0.3, 1.0):
    s = sym_fp(beta)
    print(f"beta = {beta}: symmetric state s = {s:.4f}, eigenvalues {np.linalg.eigvals(jac2(np.array([s, s]), beta)).round(4)} per ms")
print("winner state:", win.round(4), " eigenvalues", np.linalg.eigvals(jac2(win, 1.0)).round(4))

In [ ]:
import numpy as np
def test_sym():
    "the symmetric fixed point for beta = 1 is s = 0.5"
    assert abs(sym_fp(1.0) - 0.5) < 1e-8

def test_kinds():
    "the symmetric state is stable for beta = 0.3 and a saddle for beta = 1"
    e3 = np.linalg.eigvals(jac2(np.array([sym_fp(0.3)] * 2), 0.3)).real
    e1 = np.linalg.eigvals(jac2(np.array([0.5, 0.5]), 1.0)).real
    assert e3.max() < 0 and e1.min() < 0 < e1.max()

def test_winner():
    "the winner state has r1 above 0.95 and r2 below 0.05, and is stable"
    assert win[0] > 0.95 and win[1] < 0.05 and np.linalg.eigvals(jac2(win, 1.0)).real.max() < 0

test_sym()
test_kinds()
test_winner()
print("All checks passed")

### Exercise: Part B: decision time and the unstable eigenvalue

Compute `lam_plus`, the positive eigenvalue at the saddle (beta = 1). For each delta, integrate with RK4 (dt = 0.01 ms) until |r1 − r2| exceeds 0.5 and store the times in `T_dec`. Fit T_dec against ln(1/delta) and store the slope in `slope_B`.

In [ ]:
lam_plus = ...
deltas = np.array([1e-2, 1e-3, 1e-4, 1e-5, 1e-6])
T_dec = []
for d in deltas:
    r, t, dt = np.array([0.5 + d, 0.5 - d]), 0.0, 0.01
    while abs(r[0] - r[1]) < ...:
        r = rk4(lambda x: dec(x, 1.0), r, dt); t += dt
    T_dec.append(t)
T_dec = np.array(T_dec)
slope_B = np.polyfit(np.log(1 / deltas), T_dec, 1)[0]
print("decision times (ms):", T_dec.round(2), f";  slope {slope_B:.3f} ms, 1/lambda+ = {1 / lam_plus:.3f} ms")
plt.semilogx(deltas, T_dec, "o-"); plt.xlabel("initial difference delta"); plt.ylabel("decision time (ms)"); plt.show()

In [ ]:
import numpy as np
def test_slope():
    "the slope of decision time against ln(1/delta) is within 3% of 1 / lambda_plus"
    assert abs(slope_B - 1 / lam_plus) * lam_plus < 0.03

def test_order():
    "smaller tips take longer"
    assert np.all(np.diff(T_dec) > 0)

test_slope()
test_order()
print("All checks passed")

In [ ]:
gNa, gK, gL, ENa, EK, EL, C = 120.0, 36.0, 0.3, 50.0, -77.0, -54.387, 1.0
def hh_rates(V):
    an = 0.01 * (V + 55) / (1 - np.exp(-(V + 55) / 10)); bn = 0.125 * np.exp(-(V + 65) / 80)
    am = 0.1 * (V + 40) / (1 - np.exp(-(V + 40) / 10)); bm = 4.0 * np.exp(-(V + 65) / 18)
    ah = 0.07 * np.exp(-(V + 65) / 20); bh = 1 / (1 + np.exp(-(V + 35) / 10))
    return an, bn, am, bm, ah, bh
def hh_rhs(y, I):
    V, n, m, h = y
    an, bn, am, bm, ah, bh = hh_rates(V)
    dV = (I - gNa * m ** 3 * h * (V - ENa) - gK * n ** 4 * (V - EK) - gL * (V - EL)) / C
    return np.array([dV, an * (1 - n) - bn * n, am * (1 - m) - bm * m, ah * (1 - h) - bh * h])
def steady(V):
    an, bn, am, bm, ah, bh = hh_rates(V)
    return np.array([V, an / (an + bn), am / (am + bm), ah / (ah + bh)])
print("dV/dt at V = -65 with gates at steady state, I = 0:", round(hh_rhs(steady(-65.0), 0.0)[0], 4))

### Exercise: Part C: the Hopf point of Hodgkin-Huxley

Write `hh_rest(I)` (bisection for V in [−75, −50] on dV/dt of steady(V)) and `hh_jac(y, I)` (central differences, step 1e-6). Scan I from 0 to 12 in steps of 0.02, and store in `I_crit` the first current where the largest real part of an eigenvalue is positive, and in `f_crit` the frequency (Hz) of that eigenvalue.

In [ ]:
def hh_rest(I):
    g = lambda V: hh_rhs(steady(V), I)[0]
    lo, hi = -75.0, -50.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if g(lo) * g(mid) <= 0: hi = mid
        else: lo = mid
    return steady((lo + hi) / 2)

def hh_jac(y, I, h=1e-6):
    J = np.zeros((4, 4))
    for j in range(4):
        e = np.zeros(4); e[j] = h
        J[:, j] = ...
    return J

I_crit = f_crit = None; top = []
for I in np.arange(0, 12.0001, 0.02):
    ev = np.linalg.eigvals(hh_jac(hh_rest(I), I))
    k = np.argmax(ev.real); top.append(ev[k].real)
    if ev[k].real > 0 and I_crit is None:
        I_crit, f_crit = ..., ...                  # time is in ms
print(f"rest at I = 0: {hh_rest(0.0)[0]:.3f} mV;  rest loses stability at I = {I_crit:.2f} uA/cm2, frequency {f_crit:.1f} Hz")
plt.plot(np.arange(0, 12.0001, 0.02), top); plt.axhline(0, c="k", lw=0.8); plt.xlabel("I"); plt.ylabel("largest Re(eigenvalue)"); plt.show()

In [ ]:
import numpy as np
def test_rest():
    "the resting voltage at I = 0 is within 0.05 mV of -65"
    assert abs(hh_rest(0.0)[0] + 65) < 0.05

def test_crit():
    "rest loses stability between 9.5 and 10 uA/cm2, through a complex pair (frequency 80 to 110 Hz)"
    assert 9.5 < I_crit < 10.0 and 80 < f_crit < 110

test_rest()
test_crit()
print("All checks passed")